In [1]:
import pandas as pd

# 1. Load the updated BRAX master spreadsheet
df = pd.read_csv('master_spreadsheet_update.csv')

/var/folders/0y/xzwy9cz54g9b7ghtkf3shzlh0000gn/T/ipykernel_24017/2425176117.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# 2. Parse StudyDate and sort chronologically
df['StudyDate'] = pd.to_datetime(df['StudyDate'])
df = df.sort_values('StudyDate').reset_index(drop=True)

print(f"Total studies: {len(df)} | Total unique patients: {df['PatientID'].nunique()}")

Total studies: 40967 | Total unique patients: 18442


In [3]:
# 3. Chronological 60/20/20 Split by Studies
total_rows = len(df)
train_cutoff = int(total_rows * 0.60)
val_cutoff = int(total_rows * 0.80)

train_df = df.iloc[:train_cutoff].copy()
val_df = df.iloc[train_cutoff:val_cutoff].copy()
test_df = df.iloc[val_cutoff:].copy()

In [8]:
# 4. Strict Leakage Prevention
train_patients = set(train_df['PatientID'])
val_patients = set(val_df['PatientID'])
test_patients = set(test_df['PatientID'])

print(f"\nLeakage Check (Train/Val overlap): {len(train_patients.intersection(val_patients))}")
print(f"Leakage Check (Train/Test overlap): {len(train_patients.intersection(test_patients))}")
print(f"Leakage Check (Val/Test overlap): {len(val_patients.intersection(test_patients))}")


Leakage Check (Train/Val overlap): 0
Leakage Check (Train/Test overlap): 0
Leakage Check (Val/Test overlap): 0


In [5]:
# 5. Tag and save manifests
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

final_manifest = pd.concat([train_df, val_df, test_df], ignore_index=True)
final_manifest.to_csv('brax_temporal_manifest_updated.csv', index=False)

In [6]:
final_manifest.head()

,Unnamed: 0,DicomPath,PngPath,PatientID,PatientSex,PatientAge,AccessionNumber,StudyDate,No Finding,Enlarged Cardiomediastinum,...,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,ViewPosition,Rows,Columns,Manufacturer,split
0,3839,Anonymized_DICOMs/id_dc076983-9b82dd59-beaf0f0...,images/id_dc076983-9b82dd59-beaf0f0c-bc580949-...,id_dc076983-9b82dd59-beaf0f0c-bc580949-653436f3,M,0,7937928,1970-01-01 00:00:00.020080325,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,L,1535,1440,1,train
1,6484,Anonymized_DICOMs/id_dc076983-9b82dd59-beaf0f0...,images/id_dc076983-9b82dd59-beaf0f0c-bc580949-...,id_dc076983-9b82dd59-beaf0f0c-bc580949-653436f3,M,0,7937928,1970-01-01 00:00:00.020080325,0.0,NaN,...,NaN,0.0,NaN,0.0,NaN,AP,1341,1681,1,train
2,26346,Anonymized_DICOMs/id_410d3864-43453f06-b2fb046...,images/id_410d3864-43453f06-b2fb0461-fefefbc4-...,id_410d3864-43453f06-b2fb0461-fefefbc4-e08ea0c5,F,25,21084879,1970-01-01 00:00:00.020080327,1.0,0.0,...,NaN,0.0,NaN,0.0,NaN,NaN,2377,1687,5,train
3,29173,Anonymized_DICOMs/id_410d3864-43453f06-b2fb046...,images/id_410d3864-43453f06-b2fb0461-fefefbc4-...,id_410d3864-43453f06-b2fb0461-fefefbc4-e08ea0c5,F,25,21084879,1970-01-01 00:00:00.020080327,1.0,0.0,...,NaN,0.0,NaN,0.0,NaN,NaN,2103,2277,5,train
4,9375,Anonymized_DICOMs/id_eee3d599-6785c47e-0cbef87...,images/id_eee3d599-6785c47e-0cbef87b-523cfd91-...,id_eee3d599-6785c47e-0cbef87b-523cfd91-1288b693,F,30,90728730,1970-01-01 00:00:00.020080328,1.0,0.0,...,NaN,0.0,NaN,NaN,NaN,L,2809,1991,5,train
